In [11]:
!pip uninstall -y tensorflow
!pip cache purge

Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0


ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\CHANUKA ATHALAGE\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_internal\cli\base_command.py", line 180, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "C:\Users\CHANUKA ATHALAGE\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_internal\commands\uninstall.py", line 110, in run
    uninstall_pathset.commit()
  File "C:\Users\CHANUKA ATHALAGE\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_internal\req\req_uninstall.py", line 432, in commit
    self._moved_paths.commit()
  File "C:\Users\CHANUKA ATHALAGE\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_internal\req\req_uninstall.py", line 278, in commit
    save_dir.cleanup()
  File "C:\Users\CHANUKA ATHALAGE\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_internal\utils\temp_dir.py", line 173, in cleanup
    rmtree(self._path)
  File "C:\Users\CHANUKA ATH

Files removed: 322


In [1]:
!pip install tensorflow==2.15.0

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0)
ERROR: No matching distribution found for tensorflow==2.15.0

[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: C:\Users\CHANUKA ATHALAGE\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


# Load Data

In [3]:
import numpy as np

loaded = np.load("ecg_dataset.npz")
X = loaded["x"]
y = loaded["y"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Model Creation

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


def build_cnn_model(input_shape=(5000, 12)): # (time_steps, channels)
    model = models.Sequential([
        layers.Conv1D(64, 5, activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Conv1D(256, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(128, activation='relu'),
    ])

    return model

cnn_model = build_cnn_model()  
cnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])


In [8]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense

# Clone base
feature_extractor = cnn_model

# Add output layer (e.g., binary classification for heart failure)
output = Dense(1, activation='sigmoid')(feature_extractor.output)
training_model = Model(inputs=feature_extractor.input, outputs=output)

training_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train Model

In [9]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
        monitor='val_loss',       # Track validation loss
        patience=3,               # Stop after 3 epochs with no improvement
        restore_best_weights=True
    )
    

In [ ]:
training_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop])

Epoch 1/100
340/340 [==============================] - 301s 881ms/step - loss: 0.2827 - accuracy: 0.8840 - val_loss: 0.2817 - val_accuracy: 0.8801
Epoch 2/100
340/340 [==============================] - 301s 885ms/step - loss: 0.2496 - accuracy: 0.8966 - val_loss: 0.3053 - val_accuracy: 0.8801
Epoch 3/100
340/340 [==============================] - 299s 878ms/step - loss: 0.2457 - accuracy: 0.8973 - val_loss: 0.2921 - val_accuracy: 0.8842
Epoch 4/100
340/340 [==============================] - 293s 862ms/step - loss: 0.2367 - accuracy: 0.9020 - val_loss: 0.2905 - val_accuracy: 0.8941
